# Self-Supervised Fisheye Rectification Training on NYU Depth V2

This notebook trains a fisheye rectification model using the NYU Depth V2 dataset.
- Supports stop and resume from last epoch
- Plots train and validation loss
- Includes inference for undistorting images

In [ ]:
# Install required dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -q
!pip install opencv-python-headless pillow scipy pyyaml matplotlib tqdm -q

In [ ]:
# Clone the repository
import os
os.chdir('/kaggle/working')
!git clone https://github.com/memara111/SelfSupervisedFisheyeRectification.git
os.chdir('/kaggle/working/SelfSupervisedFisheyeRectification')

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU count: {torch.cuda.device_count()}')
    print(f'GPU name: {torch.cuda.get_device_name(0)}')

In [ ]:
# Setup paths
DATA_PATH = '/kaggle/input/datasets/soumikrakshit/nyu-depth-v2/nyu_data/data/nyu2_train'
OUTPUT_DIR = '/kaggle/working/outputs/nyu_depth_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Data path: {DATA_PATH}')
print(f'Output dir: {OUTPUT_DIR}')
print(f'Data exists: {os.path.exists(DATA_PATH)}')

In [ ]:
# Explore dataset structure
import os
subfolders = [f for f in os.listdir(DATA_PATH) if os.path.isdir(os.path.join(DATA_PATH, f))]
print(f'Number of subfolders: {len(subfolders)}')
print(f'First 10 subfolders: {subfolders[:10]}')

# Check what's inside a subfolder
if len(subfolders) > 0:
    first_folder = subfolders[0]
    files = os.listdir(os.path.join(DATA_PATH, first_folder))
    print(f'\nFiles in {first_folder}: {files[:10]}')

In [ ]:
# Prepare dataset - create list files for train/val split
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
import random

# Collect all RGB images (not depth)
image_paths = []
for folder in subfolders:
    folder_path = os.path.join(DATA_PATH, folder)
    for file in os.listdir(folder_path):
        # NYU Depth V2 has both color images and depth maps
        # Color images typically don't have 'depth' in the name
        if file.endswith('.jpg') or file.endswith('.png'):
            if 'depth' not in file.lower() and 'sync' not in file.lower():
                image_paths.append(os.path.join(folder_path, file))

print(f'Total images found: {len(image_paths)}')

# Split into train and validation
train_paths, val_paths = train_test_split(image_paths, test_size=0.1, random_state=42)
print(f'Train images: {len(train_paths)}')
print(f'Validation images: {len(val_paths)}')

# Write list files
data_dir = '/kaggle/working/SelfSupervisedFisheyeRectification/data/nyu_depth_v2'
os.makedirs(data_dir, exist_ok=True)

with open(os.path.join(data_dir, 'train.lst'), 'w') as f:
    for path in train_paths:
        f.write(path + '\n')

with open(os.path.join(data_dir, 'val.lst'), 'w') as f:
    for path in val_paths:
        f.write(path + '\n')

print(f'Created train.lst with {len(train_paths)} images')
print(f'Created val.lst with {len(val_paths)} images')

In [ ]:
# Create config file for NYU dataset
config_content = """
DATASET:
  NAME: nyu_depth_v2
  HEIGHT: 256
  WIDTH: 512
TRAIN:
  MAX_EPOCH: 50
  BATCH_SIZE: 8
  LEARNING_RATE: 0.0001
  CURRICULUM:
    ENABLED: true
    SWITCH_EPOCH: 10
TEST:
  CHECKPOINT: checkpoint.pth.tar
  SAVE_RESULTS: true
  OUTPUT_SIZE:
    HEIGHT: 256
    WIDTH: 512
"""

with open('/kaggle/working/SelfSupervisedFisheyeRectification/cfg/nyu_depth_v2.yml', 'w') as f:
    f.write(config_content)

print('Config file created at cfg/nyu_depth_v2.yml')

In [ ]:
# Import modules and setup
import sys
sys.path.insert(0, '/kaggle/working/SelfSupervisedFisheyeRectification/src')

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from models import ParametersEstimationModule
from core.functions import train, val, getDistortions
from core.losses import DistortionLoss
from core.config import Config
from datasets import DistortDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {DEVICE}')

In [ ]:
# Load configuration
config = Config('/kaggle/working/SelfSupervisedFisheyeRectification/cfg/nyu_depth_v2.yml').getDict()
print('Configuration loaded:')
print(f"  Dataset: {config['DATASET']['NAME']}")
print(f"  Image size: {config['DATASET']['WIDTH']}x{config['DATASET']['HEIGHT']}")
print(f"  Max epochs: {config['TRAIN']['MAX_EPOCH']}")
print(f"  Batch size: {config['TRAIN']['BATCH_SIZE']}")
print(f"  Learning rate: {config['TRAIN']['LEARNING_RATE']}")

In [ ]:
# Initialize model
in_channels = 3
model = ParametersEstimationModule(in_channels=in_channels).to(DEVICE)
transform = model.getTransforms()

if DEVICE == 'cuda':
    model = torch.nn.DataParallel(model)

criterion = DistortionLoss().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=config['TRAIN']['LEARNING_RATE'])

print('Model initialized successfully')

In [ ]:
# Load checkpoint if exists (for resuming training)
max_epoch = config['TRAIN']['MAX_EPOCH']
last_epoch = 0
train_losses = []
val_losses = []
enable_curriculum = config['TRAIN']['CURRICULUM']['ENABLED']
switch_epoch = config['TRAIN']['CURRICULUM']['SWITCH_EPOCH']

model_state_file = os.path.join(OUTPUT_DIR, 'checkpoint.pth.tar')

if os.path.exists(model_state_file):
    checkpoint = torch.load(model_state_file)
    last_epoch = checkpoint['epoch']
    train_losses = checkpoint['train_losses']
    val_losses = checkpoint['val_losses']
    model.load_state_dict(checkpoint['state_dict'], strict=False)
    optimizer.load_state_dict(checkpoint['optimizer'])
    print(f'=> Resumed from checkpoint (epoch {last_epoch})')
else:
    print('=> No checkpoint found, starting from scratch')

In [ ]:
# Setup curriculum learning distortions
if enable_curriculum:
    num_patterns = int(last_epoch / switch_epoch) + 2
    if num_patterns <= 10:
        distortions = getDistortions(num_patterns, random_values=False)
    else:
        distortions = getDistortions(10, random_values=True)
else:
    distortions = getDistortions(10, random_values=True)

print(f'Using {len(distortions)} distortion patterns')
print(f'Distortions: {distortions}')

In [ ]:
# Create datasets and dataloaders
data_path = os.path.join('/kaggle/working/SelfSupervisedFisheyeRectification/data', config['DATASET']['NAME'])

trainset = DistortDataset(
    list_path=os.path.join(data_path, 'train.lst'),
    height=config['DATASET']['HEIGHT'],
    width=config['DATASET']['WIDTH'],
    transform=transform,
    distortions=distortions,
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=config['TRAIN']['BATCH_SIZE'],
    shuffle=True,
    num_workers=2
)

testset = DistortDataset(
    list_path=os.path.join(data_path, 'val.lst'),
    height=config['DATASET']['HEIGHT'],
    width=config['DATASET']['WIDTH'],
    transform=transform,
    distortions=distortions,
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=config['TRAIN']['BATCH_SIZE'],
    shuffle=False,
    num_workers=2
)

print(f'Train samples: {len(trainset)}')
print(f'Validation samples: {len(testset)}')
print(f'Train batches: {len(trainloader)}')
print(f'Validation batches: {len(testloader)}')

In [ ]:
# Training loop with progress tracking and loss plotting
import matplotlib.pyplot as plt
from IPython.display import clear_output

print(f'Starting training from epoch {last_epoch+1} to {max_epoch}')
print('='*60)

for epoch in range(last_epoch, max_epoch):
    print(f'\nEpoch {epoch+1}/{max_epoch}')
    print('-'*40)
    
    # Training
    model = model.train()
    running_loss = 0.0
    num_iter = 0
    
    pbar = tqdm(trainloader, desc='Train')
    for inputs, labels in pbar:
        optimizer.zero_grad()
        outputs = model(inputs.to(DEVICE))
        labels = [(x.to(DEVICE), y.to(DEVICE)) for x, y in labels]
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        num_iter += 1
        pbar.set_postfix({'loss': running_loss / num_iter})
    
    train_loss = running_loss / num_iter
    
    # Validation
    model = model.eval()
    val_running_loss = 0.0
    val_num_iter = 0
    
    with torch.no_grad():
        for inputs, labels in testloader:
            outputs = model(inputs.to(DEVICE))
            labels = [(x.to(DEVICE), y.to(DEVICE)) for x, y in labels]
            loss = criterion(outputs, labels)
            val_running_loss += loss.item()
            val_num_iter += 1
    
    val_loss = val_running_loss / val_num_iter
    
    # Update losses
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f'Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}')
    
    # Plot losses
    plt.figure(figsize=(10, 5))
    plt.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss', linewidth=2)
    plt.plot(range(1, len(val_losses)+1), val_losses, label='Val Loss', linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training and Validation Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(os.path.join(OUTPUT_DIR, 'losses.png'))
    plt.show()
    
    # Curriculum learning update
    if (epoch+1) % switch_epoch == 0:
        num_patterns = int((epoch+1) / switch_epoch) + 2
        if enable_curriculum and num_patterns <= 10:
            distortions = getDistortions(num_patterns, random_values=False)
        else:
            distortions = getDistortions(10, random_values=True)
        trainset.updateEffector(distortions=distortions)
        testset.updateEffector(distortions=distortions)
        print(f'Updated curriculum: {len(distortions)} distortion patterns')
    
    # Save checkpoint
    torch.save({
        'epoch': epoch+1,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict()
    }, model_state_file)
    print(f'Checkpoint saved to {model_state_file}')

print('\n' + '='*60)
print('Training completed!')

In [ ]:
# Display final loss plot
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_losses)+1), train_losses, label='Train Loss', linewidth=2, marker='o')
plt.plot(range(1, len(val_losses)+1), val_losses, label='Val Loss', linewidth=2, marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Final Training and Validation Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig(os.path.join(OUTPUT_DIR, 'final_losses.png'), dpi=150)
plt.show()

print(f'\nFinal Train Loss: {train_losses[-1]:.6f}')
print(f'Final Val Loss: {val_losses[-1]:.6f}')